In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
start_time = 1187008869
end_time = 1187008886
seg_len = 16
sample_rate = 512
seg_start_pad = 4
seg_end_pad = 2
trigger_start = int(start_time) + seg_start_pad
trigger_end = int(end_time) - seg_end_pad
data_start = (trigger_start - seg_start_pad) - int(start_time)
data_end = trigger_end + seg_end_pad - int(start_time)
data_dur = data_end - data_start
data_start *= sample_rate
data_end *= sample_rate
analyzable = trigger_end - trigger_start
throwaway_size = seg_start_pad + seg_end_pad
seg_width = seg_len - throwaway_size

In [3]:
def calculate_segments(segment_overlap):
    #number of segments we need to analyze this data
    assert seg_width > segment_overlap, "Segment overlap is too big."
    num_segs = int(numpy.ceil(1 + float(analyzable - seg_width) 
        / float(seg_width - segment_overlap)))
    # The offset we will use between segments
    seg_offset = int(numpy.ceil(analyzable / float(num_segs)))
    segment_slices = []
    analyze_slices = []

    # Determine how to chop up the strain into smaller segments
    for nseg in range(num_segs-1):
        # boundaries for time slices into the strain
        seg_start = int(data_start + nseg * (seg_offset - segment_overlap) * sample_rate)
        seg_end = int(seg_start + seg_len * sample_rate)
        seg_slice = slice(seg_start, seg_end)
        segment_slices.append(seg_slice)

        # boundaries for the analyzable portion of the segment
        ana_start = int(seg_start_pad * sample_rate)
        ana_end = int(ana_start + seg_offset * sample_rate)
        ana_slice = slice(ana_start, ana_end)
        analyze_slices.append(ana_slice)

    # The last segment takes up any integer boundary slop
    seg_end = int(data_end)
    seg_start = int(seg_end - seg_len * sample_rate)
    seg_slice = slice(seg_start, seg_end)
    segment_slices.append(seg_slice)

    remaining = (data_dur - ((num_segs - 1) * (seg_offset - segment_overlap) + seg_start_pad))
    ana_start = int((seg_len - remaining) * sample_rate)
    ana_end = int((seg_len - seg_end_pad) * sample_rate)
    ana_slice = slice(ana_start, ana_end)
    analyze_slices.append(ana_slice)
    analyze_lengths = [asl.stop - asl.start for asl in analyze_slices]
    total_analyzed = sum(analyze_lengths) / sample_rate - (num_segs-1) * segment_overlap
    print(total_analyzed, analyzable)
    assert total_analyzed == analyzable, f"Only {total_analyzed}/{analyzable} seconds will be analyzed."
    return analyze_slices, segment_slices
analyze_slices, segment_slices = calculate_segments(segment_overlap=2000/sample_rate)
data = linspace(0, end_time-start_time, (end_time-start_time)*sample_rate)
analyzable_data = linspace(trigger_start-start_time, trigger_end-start_time, analyzable*sample_rate)
analyze_slices, segment_slices

11.0 11


([slice(2048, 5120, None), slice(2608, 7168, None)],
 [slice(0, 8192, None), slice(512, 8704, None)])

In [5]:
calculate_segments(segment_overlap=1500/sample_rate)

11.0 11


([slice(2048, 5120, None), slice(3108, 7168, None)],
 [slice(0, 8192, None), slice(512, 8704, None)])

In [6]:
calculate_segments(segment_overlap=17/sample_rate)

11.0 11


([slice(2048, 5120, None), slice(4591, 7168, None)],
 [slice(0, 8192, None), slice(512, 8704, None)])

In [7]:
calculate_segments(segment_overlap=0)

11.0 11


([slice(2048, 5120, None), slice(4608, 7168, None)],
 [slice(0, 8192, None), slice(512, 8704, None)])